In [ ]:
# ==========================================
# IMPORTURI ȘI CURĂȚARE MEMORIE
# ==========================================
import os
import time
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def curata_memoria():
    print("Începem curățarea memoriei...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("✔ Memoria RAM și VRAM este acum curată!\n")

curata_memoria()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rulăm pe: {device} | GPU-uri paralele: {torch.cuda.device_count()}\n")

# ==========================================
# 1. GENERAREA DATELOR PENTRU FIECARE SET
# ==========================================
def get_dataset_paths(dataset_name):
    if dataset_name == "APTOS":
        base = "/home/marian-s/Disertatie/Datasets/Dataset_APTOS_2019"
        return {"csv": f"{base}/train_1.csv", "dir": f"{base}/train_images/train_images"}
    elif dataset_name == "EYEPACS":
        base = "/home/marian-s/Disertatie/Datasets/Dataset_KaggleEyePACS"
        return {"csv": f"{base}/labels/traintestLabels15_trainLabels19.csv", "dir": f"{base}/resized_traintest15_train19"}
    elif dataset_name == "MESSIDOR":
        base = "/home/marian-s/Disertatie/Datasets/Dataset_Messidor_2"
        return {"csv": f"{base}/messidor_data.csv", "dir": f"{base}/messidor-2/messidor-2/preprocess"}
    elif dataset_name == "IDRID":
        base = "/home/marian-s/Disertatie/Datasets/Dataset_IDRID"
        return {"csv": f"{base}/idrid_labels.csv", "dir": f"{base}/Imagenes/Imagenes"}
    return None

def build_dataframe_for_dataset(ds_name, max_samples=None):
    """Citește CSV-ul, corectează extensiile și returnează un DataFrame curat [path, label]"""
    paths = get_dataset_paths(ds_name)
    df = pd.read_csv(paths["csv"])
    
    lista_date = []
    for idx in range(len(df)):
        img_name = str(df.iloc[idx, 0])
        base_name = os.path.splitext(img_name)[0]
        
        if ds_name == "EYEPACS": ext = '.jpeg'
        elif ds_name == "APTOS": ext = '.png'
        elif ds_name == "MESSIDOR": ext = '.png'
        elif ds_name == "IDRID": ext = '.jpg'
            
        full_path = os.path.join(paths["dir"], base_name + ext)
        label_binar = 0 if int(df.iloc[idx, 1]) == 0 else 1
        
        lista_date.append({'path': full_path, 'label': label_binar})
        
    df_final = pd.DataFrame(lista_date)
    
    # Dacă setul este prea mare (ex: EyePACS), îl eșantionăm pentru a păstra experimentul rapid
    if max_samples and len(df_final) > max_samples:
        print(f"  [!] Setul {ds_name} are {len(df_final)} imagini. Eșantionăm la {max_samples}...")
        df_final = df_final.sample(n=max_samples, random_state=42)
        
    return df_final

# ==========================================
# 2. DATASET BUILDER
# ==========================================
class DataFrameRetinopathyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        path = self.dataframe.iloc[idx]['path']
        label = torch.tensor(self.dataframe.iloc[idx]['label'], dtype=torch.float32)
        
        # Siguranță
        if not os.path.exists(path):
            baza = os.path.splitext(path)[0]
            for ext in ['.png', '.jpeg', '.jpg', '.JPG', '.PNG', '.JPEG']:
                if os.path.exists(baza + ext):
                    path = baza + ext
                    break

        try:
            image = Image.open(path).convert('RGB')
        except FileNotFoundError:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform: 
            image = self.transform(image)
            
        return image, label

# Transformări (Folosim o augmentare ușoară pentru a preveni overfitting-ul pe IDRID)
transforms_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transforms_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==========================================
# 3. SELECTOR DE MODEL (ResNet50 cu Deep Fine-Tuning)
# ==========================================
def build_model():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for param in model.parameters(): param.requires_grad = False
    
    # Dezghețăm 'layer4' pentru extragerea texturilor vasculare
    for param in model.layer4.parameters(): param.requires_grad = True
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 1))
    
    # Trucul Karpathy pentru un start mai bun al Loss-ului
    pi = 0.3 # Estimare brută a proporției de bolnavi
    bias_initial = -np.log((1 - pi) / pi)
    model.fc[1].bias.data.fill_(bias_initial)
    
    return model

# ==========================================
# 4. EXECUTAREA EXPERIMENTELOR (CELE 4 SETURI)
# ==========================================
SETURI_DE_DATE = ["APTOS", "IDRID", "MESSIDOR", "EYEPACS"]
EPOCI = 15
LR = 0.0001 # Un Learning Rate mic este ideal pentru Deep Fine-Tuning

for ds_name in SETURI_DE_DATE:
    print(f"\n{'='*65}")
    print(f"🚀 EVALUARE BASELINE: {ds_name}")
    print(f"{'='*65}")
    
    # 4.1. Pregătirea datelor (Max 15.000 imagini pentru a nu bloca sistemul)
    df_complet = build_dataframe_for_dataset(ds_name, max_samples=15000)
    
    # Împărțire internă 80% Antrenare / 20% Testare (Păstrând proporția de bolnavi cu stratify)
    train_df, test_df = train_test_split(df_complet, test_size=0.20, random_state=42, stratify=df_complet['label'])
    print(f"  -> Set împărțit: {len(train_df)} Antrenare | {len(test_df)} Testare")
    
    train_ds = DataFrameRetinopathyDataset(train_df, transform=transforms_train)
    test_ds = DataFrameRetinopathyDataset(test_df, transform=transforms_test)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=8, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=8, pin_memory=True)
    
    # 4.2. Inițializare Model
    model = build_model()
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    model = model.to(device)
    
    params_to_train = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params_to_train, lr=LR)
    criterion = nn.BCEWithLogitsLoss()
    
    istoric = {'train_loss': [], 'test_loss': [], 'test_f1': [], 'test_acc': []}
    best_f1_score = 0.0
    
    # 4.3. Bucla de antrenare
    for epoch in range(EPOCI):
        start_time = time.time()
        
        # --- TRAIN ---
        model.train()
        running_train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        # --- TEST/EVAL ---
        model.eval()
        running_test_loss = 0.0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device).unsqueeze(1)
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_test_loss += loss.item()
                
                probs = torch.sigmoid(outputs)
                all_preds.extend((probs >= 0.5).float().cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_test_loss = running_test_loss / len(test_loader)
        epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)
        epoch_acc = accuracy_score(all_labels, all_preds)
        
        istoric['train_loss'].append(avg_train_loss)
        istoric['test_loss'].append(avg_test_loss)
        istoric['test_f1'].append(epoch_f1)
        istoric['test_acc'].append(epoch_acc)
        
        m, s = divmod(time.time() - start_time, 60)
        print(f"  Ep[{epoch+1}/{EPOCI}] | T_Loss: {avg_train_loss:.4f} | V_Loss: {avg_test_loss:.4f} | F1: {epoch_f1:.4f} | Acc: {epoch_acc*100:.1f}% | Timp: {int(m)}m {int(s)}s", end="")
        
        if epoch_f1 > best_f1_score:
            best_f1_score = epoch_f1
            torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), f"Baseline_{ds_name}_best.pth")
            print(" -> Salvat!")
        else:
            print()

        del all_preds, all_labels
        gc.collect()
            
    # 4.4. Salvare Grafice
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(istoric['train_loss'], label='Train Loss', color='blue')
    ax1.plot(istoric['test_loss'], label='Val Loss', color='red', linestyle='--')
    ax1.set_title(f"Evoluție Loss: {ds_name}")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    
    ax2.plot(istoric['test_acc'], label='Accuracy', color='green')
    ax2.plot(istoric['test_f1'], label='F1-Score', color='purple', linestyle='-.')
    ax2.set_title(f"Evoluție Metrici: {ds_name}")
    ax2.legend(); ax2.grid(True, alpha=0.3)
    
    plt.savefig(f"grafic_baseline_{ds_name}.png", dpi=300)
    plt.close()
    
    print(f"✔ Finalizat experimentul pentru {ds_name}. Eliberăm memoria...")
    del model, optimizer, train_loader, test_loader
    torch.cuda.empty_cache()
    gc.collect()
    time.sleep(2)

print("\n🎉 TOATE CELE 4 SETURI DE DATE AU FOST EVALUATE CU SUCCES! 🎉")

In [5]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

# ==========================================
# 1. SETĂRI GLOBALE
# ==========================================
BASE_DIR = "./Datasets"
DATASETS = ["idrid", "aptos", "kaggleeyepacs", "messidor-2"]

BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 5

# Setăm device-ul (GPU dacă este disponibil, altfel CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Folosim device-ul: {device}")

# Transformări specifice pentru ResNet50 (pre-antrenat pe ImageNet)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(), # Data augmentation simplu
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# ==========================================
# 2. FUNCȚII AJUTĂTOARE
# ==========================================
def build_resnet50(num_classes):
    """Încarcă ResNet50, îngheață straturile de bază și modifică ultimul strat."""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    
    # Înghețăm parametrii (Transfer Learning)
    for param in model.parameters():
        param.requires_grad = False
        
    # Înlocuim capătul de clasificare (fc = fully connected)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, num_classes)
    )
    
    return model.to(device)

def plot_and_save_history(history, dataset_name):
    """Salvează graficele de performanță."""
    epochs = range(1, len(history['train_acc']) + 1)
    
    plt.figure(figsize=(12, 5))
    
    # Grafic Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_acc'], label='Train Accuracy')
    plt.plot(epochs, history['val_acc'], label='Validation Accuracy')
    plt.title(f'{dataset_name} - Accuracy')
    plt.legend()
    
    # Grafic Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Validation Loss')
    plt.title(f'{dataset_name} - Loss')
    plt.legend()
    
    plt.savefig(f"{dataset_name}_training_history.png")
    plt.close()

# ==========================================
# 3. BUCLA DE ANTRENAMENT (Custom PyTorch)
# ==========================================
def train_model(model, dataloaders, criterion, optimizer, dataset_name, num_epochs=15, patience=3):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Fiecare epocă are o fază de antrenament și una de validare
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Iterăm prin date
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + Optimizare doar în antrenament
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            # Salvăm istoricul
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())

                # Early Stopping & Salvare model
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), f'best_model_{dataset_name}.pth')
                else:
                    epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f'Early stopping declanșat după {epoch+1} epoci.')
            break
            
        print()

    print(f'Cel mai bun Validation Accuracy: {best_acc:4f}')
    # Încărcăm cele mai bune greutăți
    model.load_state_dict(best_model_wts)
    return model, history

# ==========================================
# 4. EXECUTAREA PENTRU FIECARE SET DE DATE
# ==========================================
for dataset in DATASETS:
    print(f"\n{'='*40}")
    print(f"Set de date: {dataset.upper()}")
    print(f"{'='*40}")
    
    data_dir = os.path.join(BASE_DIR, dataset)
    
    if not os.path.exists(os.path.join(data_dir, 'train')):
        print(f"[Avertisment] Nu găsesc structura pentru {dataset}. Trec peste...")
        continue
        
    # Încărcăm datele
    image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                      for x in ['train', 'val']}
    
    dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                                                 shuffle=True, num_workers=4)
                   for x in ['train', 'val']}
    
    # Inițializăm modelul, funcția de loss și optimizatorul
    model = build_resnet50(NUM_CLASSES)
    criterion = nn.CrossEntropyLoss()
    # Antrenăm doar ultimul strat (capătul de clasificare)
    optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
    
    # Antrenament
    best_model, history = train_model(model, dataloaders, criterion, optimizer, dataset, num_epochs=EPOCHS)
    
    # Grafice
    plot_and_save_history(history, dataset)

Folosim device-ul: cuda:0

Set de date: IDRID
[Avertisment] Nu găsesc structura pentru idrid. Trec peste...

Set de date: APTOS
[Avertisment] Nu găsesc structura pentru aptos. Trec peste...

Set de date: KAGGLEEYEPACS
[Avertisment] Nu găsesc structura pentru kaggleeyepacs. Trec peste...

Set de date: MESSIDOR-2
[Avertisment] Nu găsesc structura pentru messidor-2. Trec peste...
